# ATLAS Solar Scenario Preprocessing Tutorial

This notebook preprocesses CMIP6 scenario data downloaded from ESGF for the ATLAS solar workflow.

The notebook is organised as a tutorial for non expert Python users:

1. **User input parameters**: edit only this section.
2. **Helper functions**: run these cells without editing them.
3. **Run preprocessing**: execute the workflow and save the processed NetCDF file.

The workflow follows the same philosophy used for the reanalysis notebooks: paths are relative, anonymous, and country specific.


## 1. User input parameters

Edit the values below according to the country, model, experiment and variable you want to process.

The expected input folder is the output of the scenario download notebook:

`../data/esgf_download/{model}/{experiment}/{variable}/{country}/`

The processed output will be saved in:

`../data/processed/{variable}/{country}/{model}/{experiment}/`


In [1]:
from pathlib import Path

# Country name used in folder names and geographic subsetting.
# Use lowercase names, for example: "argentina", "bolivia", "colombia", "ecuador", "peru".
country = "argentina"

# CMIP6 model name.
model = "CNRM-ESM2-1"

# CMIP6 experiment.
# Typical values are: "historical", "ssp126", "ssp245", "ssp370", "ssp585".
experiment = "historical"

# Variable used in ESGF files and folder names.
# For solar radiation, this is usually "rsds".
variable = "rsds"

# Name used for the output NetCDF variable if needed.
# Keep it equal to variable unless you explicitly want to rename it later in the pipeline.
output_variable_name = variable

# Temporal aggregation mode.
# Use "daymean" to aggregate sub daily data to daily mean values.
# Use None if the files are already daily.
aggregation_mode = "daymean"

# Shapefile used only to retrieve country geometries if needed by future extensions.
# The current workflow uses the bounding boxes below for fast and simple subsetting.
shapefile_path = Path("../world_map/ne_50m_admin_0_countries.shp")

# Input and output paths.
input_path = Path(f"../data/esgf_downloads/{model}/{experiment}/{variable}/")
output_path = Path(f"../data/processed/{variable}/{country}/{model}/{experiment}/")
output_path.mkdir(parents=True, exist_ok=True)

# Country bounding boxes: [north, west, south, east].
# A small margin is added automatically during preprocessing.
areas = {
    "bolivia": [-9.6, -69.8, -23.0, -57.4],
    "argentina": [-21.7, -73.6, -55.1, -53.5],
    "ecuador": [1.9, -92.0, -5.3, -75.1],
    "peru": [0.1, -81.5, -18.5, -68.5],
    "colombia": [15.9, -81.7, -5.1, -65.9],
}

area = areas[country]

# Time range used for each experiment type.
if experiment == "historical":
    start_date = "1985-01"
    end_date = "2014-12"
else:
    start_date = "2015-01"
    end_date = "2100-12"

print(f"Input folder:  {input_path}")
print(f"Output folder: {output_path}")
print(f"Period:        {start_date} to {end_date}")


Input folder:  ../data/esgf_downloads/CNRM-ESM2-1/ssp370/clt
Output folder: ../data/processed/clt/argentina/CNRM-ESM2-1/ssp370
Period:        2015-01 to 2100-12


## 2. Imports

Run this cell before running the helper functions.


In [2]:
import os
import glob

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd

try:
    import rioxarray  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "This notebook requires rioxarray. Install it before running the workflow."
    ) from exc


## 3. Helper functions

These functions prepare coordinates, load CMIP6 files, subset the country area, aggregate data if requested and save the processed NetCDF output.


In [3]:
def roll_longitudes(ds, lon_name="longitude"):
    """Convert longitudes from 0..360 to -180..180 and sort them."""
    ds = ds.assign_coords({lon_name: ((ds[lon_name] + 180) % 360) - 180})
    return ds.sortby(lon_name)


def ensure_epsg4326(ds):
    """Assign EPSG:4326 CRS to datasets with longitude and latitude coordinates."""
    ds = ds.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
    if ds.rio.crs is None:
        ds = ds.rio.write_crs("EPSG:4326", inplace=False)
    return ds


def fix_coordinates(ds):
    """Standardise coordinate names, CRS and longitude convention."""
    rename_map = {}
    if "lon" in ds.coords:
        rename_map["lon"] = "longitude"
    if "lat" in ds.coords:
        rename_map["lat"] = "latitude"
    if rename_map:
        ds = ds.rename(rename_map)

    ds = ensure_epsg4326(ds)
    ds["longitude"] = np.round(ds.longitude, 3)
    ds["latitude"] = np.round(ds.latitude, 3)

    if float(ds.longitude.min()) >= 0:
        ds = roll_longitudes(ds)

    return ds


def preprocess_cmip_file(ds, experiment, start_date, end_date):
    """Preprocess each CMIP6 file before concatenation."""
    ds = fix_coordinates(ds)

    for name in ["time_bounds", "time_bnds", "lat_bnds", "lon_bnds", "height"]:
        if name in ds.variables:
            ds = ds.drop_vars(name)

    if "valid_time" in ds.coords:
        ds = ds.rename({"valid_time": "time"})

    ds = ds.sel(time=slice(start_date, end_date))

    for coord in ["member_id", "dcpp_init_year"]:
        if coord in ds.coords or coord in ds.dims:
            ds = ds.mean(coord)

    return ds


def load_cmip_data(input_path, experiment, start_date, end_date):
    """Load all NetCDF files from the input folder."""
    files = sorted(Path(input_path).glob("*.nc"))
    if not files:
        raise FileNotFoundError(f"No NetCDF files found in: {input_path}")

    return xr.open_mfdataset(
        files,
        combine="by_coords",
        preprocess=lambda ds: preprocess_cmip_file(ds, experiment, start_date, end_date),
    )


def subset_country_area(ds, area, margin_degrees=2.5):
    """Subset the dataset around the selected country bounding box."""
    north, west, south, east = area
    lon_min = west - margin_degrees
    lon_max = east + margin_degrees
    lat_min = south - margin_degrees
    lat_max = north + margin_degrees

    ds = ds.sortby("latitude")
    return ds.sel(longitude=slice(lon_min, lon_max), latitude=slice(lat_min, lat_max))


def aggregate_to_daily_mean(ds):
    """Aggregate the dataset to daily mean values."""
    return ds.resample(time="1D").mean()


def save_netcdf(ds, output_filename, variable_name=None, overwrite=True):
    """Save a Dataset or DataArray to NetCDF using simple and fast encoding."""
    output_filename = Path(output_filename)
    output_filename.parent.mkdir(parents=True, exist_ok=True)

    if output_filename.exists() and not overwrite:
        print(f"File already exists and overwrite is False: {output_filename}")
        return output_filename

    if isinstance(ds, xr.DataArray):
        ds = ds.rename(variable_name or ds.name or "variable").to_dataset()
    elif not isinstance(ds, xr.Dataset):
        raise TypeError("Input must be an xarray Dataset or DataArray.")

    if "spatial_ref" in ds.variables:
        ds = ds.drop_vars("spatial_ref")

    encoding = {}
    for data_var in ds.data_vars:
        enc = {"zlib": False}
        if np.issubdtype(ds[data_var].dtype, np.floating):
            enc["dtype"] = "float32"

        shape = ds[data_var].shape
        if ds[data_var].ndim == 3:
            enc["chunksizes"] = (1, min(shape[1], 512), min(shape[2], 512))
        elif ds[data_var].ndim == 2:
            enc["chunksizes"] = (min(shape[0], 512), min(shape[1], 512))

        encoding[data_var] = enc

    ds.to_netcdf(
        output_filename,
        engine="netcdf4",
        format="NETCDF4",
        encoding=encoding,
    )

    print(f"Saved: {output_filename}")
    return output_filename


## 4. Run preprocessing

This cell executes the full workflow:

1. Load the downloaded ESGF files.
2. Standardise coordinates and time range.
3. Cut the data around the selected country.
4. Optionally aggregate to daily mean.
5. Save the processed NetCDF file.


In [4]:
# Step 1: load CMIP6 scenario data from the ESGF download folder.
ds = load_cmip_data(
    input_path=input_path,
    experiment=experiment,
    start_date=start_date,
    end_date=end_date,
)

# Step 2: spatial subset around the selected country.
ds = subset_country_area(ds, area)

# Step 3: optional temporal aggregation.
if aggregation_mode == "daymean":
    ds = aggregate_to_daily_mean(ds)

# Step 4: save the processed output.
output_filename = output_path / f"{variable}_{start_date}_{end_date}_processed.nc"
save_netcdf(
    ds,
    output_filename=output_filename,
    variable_name=output_variable_name,
    overwrite=True,
)


Saved: ../data/processed/clt/argentina/CNRM-ESM2-1/ssp370/clt_2015-01_2100-12_processed.nc


PosixPath('../data/processed/clt/argentina/CNRM-ESM2-1/ssp370/clt_2015-01_2100-12_processed.nc')

## 5. Expected output

After the run cell finishes, the processed file is available in:

`../data/processed/{variable}/{country}/{model}/{experiment}/`

This output can be used as input for the scenario downscaling notebook.
